<a href="https://colab.research.google.com/github/hanidew/WIE3007-DMW-GroupProject/blob/main/Xayne's_Random_Forest_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Random Forest**

Data Loading & Cleaning

In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# ==========================================
# 1. LOAD DATA
# ==========================================
# Load the file
df = pd.read_csv('synthetic_data_cleaned_feature_engineered.csv', sep='|', on_bad_lines='skip', engine='python')

# --- FIX: CLEAN COLUMN NAMES ---
# The file has "Log_Income,," as a header. This fixes it.
df.columns = df.columns.str.replace(',,', '').str.strip()

# Cleanup: Remove accidental empty columns (trailing commas)
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Drop rows where Target is missing
df = df.dropna(subset=['Default_Status'])
df['Default_Status'] = df['Default_Status'].astype(int)

Feature Selection

In [24]:
# ==========================================
# 2. SELECT FEATURES (STRICTLY EXISTING ONLY)
# ==========================================
features = [
    'Age',
    'Annual_Income',
    'Credit_Score',
    'Loan_Amount',
    'Loan_Term_Months',
    'DTI_Ratio',
    'Log_Income',
    'Risk_Flag',
    'Occupation',
    'Has_Asset'
]

# Create X (Features) and y (Target)
X = df[features].copy()
y = df['Default_Status']

Preprocessing

In [25]:
# ==========================================
# 3. PREPROCESSING
# ==========================================
# Identify truly numeric columns that are not handled separately
numeric_cols_for_conversion = [col for col in features if col not in ['Occupation', 'Has_Asset']]

# Convert these columns to numeric, coercing errors to NaN
for col in numeric_cols_for_conversion:
    if col in X.columns:
        X[col] = pd.to_numeric(X[col], errors='coerce')

# Encode Occupation (Text -> Numbers)
X = pd.get_dummies(X, columns=['Occupation'], drop_first=True)

# Encode Asset (Yes/No -> 1/0)
if 'Has_Asset' in X.columns:
    X['Has_Asset'] = pd.to_numeric(X['Has_Asset'].replace({'Yes': 1, 'No': 0}), errors='coerce').fillna(0)

# Fill any remaining missing values (from coerce or original NaNs) with 0
X = X.fillna(0)

Model Training

In [26]:
# ==========================================
# 4. TRAIN MODEL
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)